<a href="https://colab.research.google.com/github/FatiBuuloloo/Automated_Support_Ticket_Triage_and_Intelligent_Priority_Scoring-mini_project-009/blob/main/translating_and_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install sentence-transformers

In [ ]:
pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.2 MB/s eta 0:00:00


In [ ]:
pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=21af1f6c9db71e187e90e73a96216a56acbc590fadc70905820e8fbbd553bb98
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
import pandas as pd
import re
from deep_translator import GoogleTranslator
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

In [ ]:
url="https://raw.githubusercontent.com/FatiBuuloloo/Automated_Support_Tickets_Priority/main/dataset/ticket-dataset.csv"
df = pd.read_csv(url)
df

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28582,Performance Problem with Data Analytics Tool,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,high,en,400,Performance,IT,Tech Support,NaN,NaN,NaN,NaN,NaN
28583,Datensperrung in der Kundschaftsbetreuung,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,high,de,400,Security,IT,Tech Support,Bug,NaN,NaN,NaN,NaN
28584,Problem mit der Videokonferenz-Software heute,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,low,de,400,Bug,Performance,Network,IT,Tech Support,NaN,NaN,NaN
28585,Update Request for SaaS Platform Integration F...,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,high,en,400,Feature,IT,Tech Support,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.columns.tolist()

['subject',
 'body',
 'answer',
 'type',
 'queue',
 'priority',
 'language',
 'version',
 'tag_1',
 'tag_2',
 'tag_3',
 'tag_4',
 'tag_5',
 'tag_6',
 'tag_7',
 'tag_8']

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28587 entries, 0 to 28586
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   24749 non-null  object
 1   body      28587 non-null  object
 2   answer    28580 non-null  object
 3   type      28587 non-null  object
 4   queue     28587 non-null  object
 5   priority  28587 non-null  object
 6   language  28587 non-null  object
 7   version   28587 non-null  int64 
 8   tag_1     28587 non-null  object
 9   tag_2     28574 non-null  object
 10  tag_3     28451 non-null  object
 11  tag_4     25529 non-null  object
 12  tag_5     14545 non-null  object
 13  tag_6     5874 non-null   object
 14  tag_7     2040 non-null   object
 15  tag_8     565 non-null    object
dtypes: int64(1), object(15)
memory usage: 3.5+ MB


removing identification columns

In [ ]:
df.drop("subject",axis=1,inplace=True)

Dropping irrelevant columns

In [ ]:
data = df[['body','answer','type','queue','priority','language','version','tag_1']]

In [ ]:
data[~df["answer"].isna()].reset_index(drop=True)

,body,answer,type,queue,priority,language,version,tag_1
0,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security
1,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account
2,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product
3,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing
4,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product
...,...,...,...,...,...,...,...,...
28575,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,high,en,400,Performance
28576,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,high,de,400,Security
28577,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,low,de,400,Bug
28578,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,high,en,400,Feature


In [ ]:
object_cols = data.select_dtypes("object")
object_cols.drop(columns=["body","answer"], axis=1,inplace= True)
for item in object_cols.columns:
    print(data[item].value_counts())
    print()

type
Incident    11466
Request      8187
Problem      6012
Change       2922
Name: count, dtype: int64

queue
Technical Support                  8362
Product Support                    5252
Customer Service                   4268
IT Support                         3433
Billing and Payments               2788
Returns and Exchanges              1437
Service Outages and Maintenance    1148
Sales and Pre-Sales                 918
Human Resources                     576
General Inquiry                     405
Name: count, dtype: int64

priority
medium    11515
high      11178
low        5894
Name: count, dtype: int64

language
en    16338
de    12249
Name: count, dtype: int64

tag_1
Security          5880
Bug               5337
Feedback          3557
Feature           3081
Performance       3065
                  ... 
Recovery             1
Shipping             1
Employee             1
Infrastructure       1
Warranty             1
Name: count, Length: 116, dtype: int64



In [ ]:
data[data["language"]=="de"]

,body,answer,type,queue,priority,language,version,tag_1
0,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security
8,"Sehr geehrtes Kundensupport-Team,\n\nich hoffe...",Vielen Dank für Ihre Anfrage. Wir stellen Ihne...,Request,Technical Support,low,de,51,Documentation
9,"Sehr geehrtes Kundendienstteam,\n\nich hoffe, ...","Vielen Dank, dass Sie uns bezüglich des kürzli...",Request,Service Outages and Maintenance,high,de,51,Disruption
11,"Sehr geehrtes Support-Team,\n\nich möchte Sie ...",Vielen Dank für Ihre Kontaktaufnahme bezüglich...,Incident,Product Support,medium,de,51,Bug
25,"Sehr geehrtes Support-Team,\n\nich möchte ein ...","Vielen Dank, dass Sie sich wegen der Verbindun...",Problem,IT Support,high,de,51,Network
...,...,...,...,...,...,...,...,...
28577,"geehrte Kundensupport, ich möchte mich bei Ihn...","geehrter <name>, Ihre E-Mail enthält eine ausf...",Change,Product Support,medium,de,400,Security
28579,Please contact us to obtain information regard...,We are providing guidelines for integrating Mi...,Request,Technical Support,high,de,400,Documentation
28581,Probleme mit dem Projektmanagement-Tool in der...,"Verstanden, Sie haben Kompatibilitätsprobleme ...",Problem,Returns and Exchanges,medium,de,400,Bug
28583,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,high,de,400,Security


In [ ]:
data['body'] = data['body'].str.replace(r'\\n+', ' ', regex=True)

/tmp/ipykernel_147/2394814101.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['body'] = data['body'].str.replace(r'\\n+', ' ', regex=True)


In [ ]:
data["body"]

,body
0,"Sehr geehrtes Support-Team, ich möchte einen ..."
1,"Dear Customer Support Team, I am writing to r..."
2,"Dear Customer Support Team, I hope this messa..."
3,"Dear Customer Support Team, I hope this messa..."
4,"Dear Support Team, I hope this message reache..."
...,...
28582,The data analytics tool experiences sluggish p...
28583,"Es gab einen Datensperrungsunfall, bei dem ung..."
28584,Wichtigere Sitzungen wurden unterbrochen durch...
28585,Requesting an update on the integration featur...


In [ ]:
translator = GoogleTranslator(source="de",target="en")
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def translate(text):
    text = str(text).strip()
    detected_lang = detect(text)
    if detected_lang == "de":
        return translator.translate(text)
    else:
        return text

data["body_en"] = data['body'].apply(translate)

/tmp/ipykernel_147/1927761196.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["body_en"] = data['body'].apply(translate)


In [ ]:
data

,body,answer,type,queue,priority,language,version,tag_1,body_en
0,"Sehr geehrtes Support-Team, ich möchte einen ...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,"Dear Support Team, I would like to report a se..."
1,"Dear Customer Support Team, I am writing to r...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,"Dear Customer Support Team, I am writing to r..."
2,"Dear Customer Support Team, I hope this messa...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,"Dear Customer Support Team, I hope this messa..."
3,"Dear Customer Support Team, I hope this messa...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,"Dear Customer Support Team, I hope this messa..."
4,"Dear Support Team, I hope this message reache...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,"Dear Support Team, I hope this message reache..."
...,...,...,...,...,...,...,...,...,...
28582,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,high,en,400,Performance,The data analytics tool experiences sluggish p...
28583,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,high,de,400,Security,There was a data lockout incident involving un...
28584,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,low,de,400,Bug,More important meetings were interrupted by vi...
28585,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,high,en,400,Feature,Requesting an update on the integration featur...


In [ ]:
data.to_csv("customer_ticket_data_translated.csv")

In [ ]:
classifier = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")
def get_sentiment(text):
    result = classifier(text)[0]
    return pd.Series([result["label"], result["score"]])

data[["sentiment_label","sentiment_score"]] = data["body_en"].apply(get_sentiment)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/tmp/ipykernel_147/459989527.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[["sentiment_label","sentiment_score"]] = data["body_en"].apply(get_sentiment)


In [ ]:
data

,body,answer,type,queue,priority,language,version,tag_1,body_en,sentiment_label,sentiment_score
0,"Sehr geehrtes Support-Team, ich möchte einen ...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,"Dear Support Team, I would like to report a se...",negative,0.740054
1,"Dear Customer Support Team, I am writing to r...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,"Dear Customer Support Team, I am writing to r...",negative,0.810550
2,"Dear Customer Support Team, I hope this messa...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,"Dear Customer Support Team, I hope this messa...",positive,0.629225
3,"Dear Customer Support Team, I hope this messa...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,"Dear Customer Support Team, I hope this messa...",neutral,0.629057
4,"Dear Support Team, I hope this message reache...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,"Dear Support Team, I hope this message reache...",positive,0.664589
...,...,...,...,...,...,...,...,...,...,...,...
28582,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,high,en,400,Performance,The data analytics tool experiences sluggish p...,negative,0.658245
28583,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,high,de,400,Security,There was a data lockout incident involving un...,negative,0.828765
28584,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,low,de,400,Bug,More important meetings were interrupted by vi...,negative,0.536134
28585,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,high,en,400,Feature,Requesting an update on the integration featur...,positive,0.743706


In [ ]:
data.sentiment_label.value_counts()

,count
sentiment_label,
negative,15391
neutral,7828
positive,5368


In [ ]:
index_senti = data[data["sentiment_label"]=="negative"].index

In [ ]:
import random
data.loc[random.choice(index_senti),"body_en"]

'The marketing agency is encountering billing discrepancies related to IntelliJ IDEA and Microsoft Office. The issue may be due to overlapping subscription renewals, system errors, or payment processing issues. Efforts to reconcile payment records and contact support have not resolved the problem. Assistance is needed to resolve the matter promptly to avoid any complications.'

In [ ]:
data.to_csv("customer_support_tickets_translated+labeling.csv",index=False)